### Import dependencies

In [1]:
from helpers.sporc import (
    SPORCDataset,
    get_all_categories,
    get_main_categories,
    get_main_category,
    get_subcategories_list,
    is_main_category,
    is_subcategory,
    is_valid_category,
)
from pprint import pprint
import matplotlib.pyplot as plt
from typing import Optional
import pandas as pd
import re

In [2]:
sporc = SPORCDataset(
    local_data_dir="data",
    load_samples_only=True,
    load_turns_eagerly=True,
    show_progress=False,
)

INFO:helpers.sporc.dataset:Loading SPORC dataset from local directory: data
INFO:helpers.sporc.dataset:Loading SPORC sample dataset only.
INFO:helpers.sporc.dataset:✓ All required files found
INFO:helpers.sporc.dataset:Loading all records from local files into memory...
INFO:helpers.sporc.dataset:Loading episode_data_sample...
INFO:helpers.sporc.dataset:Loading speaker_turn_data_sample...
INFO:helpers.sporc.dataset:✓ Loaded 210,000 total records from 2 files
INFO:helpers.sporc.dataset:✓ Local dataset loaded successfully in 6.02 seconds
INFO:helpers.sporc.dataset:✓ Dataset loaded successfully with 210000 total records
INFO:helpers.sporc.dataset:Processing dataset into Podcast and Episode objects...
INFO:helpers.sporc.dataset:Separating episode data from speaker turn data...
INFO:helpers.sporc.dataset:✓ Separation completed in 0.10 seconds
INFO:helpers.sporc.dataset:  Episode records: 10,000, Speaker turn records: 200,000
INFO:helpers.sporc.dataset:Grouping episodes by podcast...
INFO:he

In [3]:
# Get all categories
all_categories = get_all_categories()
main_categories = get_main_categories()
subcategories = get_subcategories_list()

In [4]:
# Accessing podcast and episode data
podcasts = sporc.get_all_podcasts() # List of Podcast objects
episodes = sporc.get_all_episodes() # List of Episode objects


________

### Call-to-Action Detection Pipeline

Call-to-Action (CTA) language is an important feature of persuasive communication. In marketing research, CTAs are defined as explicit prompts that encourage audiences to take a desired action, such as making a purchase or clicking on a link. Studies on digital advertising describe them as message components designed to urge consumers toward a specific behavioural response. In communication and CSR research, CTAs appear as elements that ask audiences to engage with a cause or organization, for example by donating or sharing content. From the perspective of speech-act theory, these correspond to directive speech acts where the speaker aims to influence the listener’s future actions. Because CTAs reveal how podcast hosts engage audiences, promote content or sponsors, and attempt to mobilize behaviour, detecting them is valuable for understanding the communicative and commercial structure of episodes.

> <font color="green"> Sources (not thoroughly checked): </font>  
> - [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S1094996818300707?)  
> - [MDPI](https://www.mdpi.com/2071-1050/13/7/3812)  
> - [ResearchGate](https://doi.org/10.15294/lc.v15i1.26029)

For this project we aim to detect whether a **turn** contains at least one CTA. Although CTAs vary widely in form and purpose, we use a binary distinction for reliability and interpretability:

- **CTA**: at least one utterance encouraging the listener to perform a specific action (for example visiting a website, subscribing, donating, or following a link).  
- **Non-CTA**: no such directive.

Binary classification is appropriate because CTAs are relatively sparse and often ambiguous. Developing fine-grained categories, such as differentiating promotional CTAs from engagement or civic CTAs, would require extensive annotation guidelines and substantial training data. Given limited annotation capacity, fine-grained distinctions would be unreliable, whereas a binary label is robust and fully sufficient for downstream analyses.

CTA identification is challenging: some cases are ambiguous or context dependent, CTAs directed at co-hosts rather than listeners ideally should not be counted, and softly phrased CTAs can be difficult to detect. Whisper transcription errors may obscure key cues, and very long turns can embed CTAs among unrelated material. Despite these limitations, our pipeline aims to provide a practical and interpretable approach to identifying directive behaviour in podcast speech.

### Call-to-Action Detection Pipeline

Call-to-Action (CTA) language is an important feature of persuasive communication. In marketing research, CTAs are defined as explicit prompts that encourage audiences to take a desired action, such as making a purchase or clicking on a link. Studies on digital advertising describe them as message components designed to urge consumers toward a specific behavioural response. In communication and CSR research, CTAs appear as elements that ask audiences to engage with a cause or organization, for example by donating or sharing content. From the perspective of speech-act theory, these correspond to directive speech acts where the speaker aims to influence the listener’s future actions. Because CTAs reveal how podcast hosts engage audiences, promote content or sponsors, and attempt to mobilize behaviour, detecting them is valuable for understanding the communicative and commercial structure of episodes.

> <font color="green">Indicative sources (not thoroughly checked):</font>  
> - [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S1094996818300707?)  
> - [MDPI](https://www.mdpi.com/2071-1050/13/7/3812)  
> - [ResearchGate](https://doi.org/10.15294/lc.v15i1.26029)

Our goal is to determine **whether a given turn contains at least one CTA**. Although CTAs often occur at the level of individual sentences, we assign labels at the **turn level** because this is the unit used throughout our conversational analysis. A turn-level label is also more interpretable for downstream episode- and speaker-level summaries. At the same time, very long turns in our dataset can exceed LLM context limits, and CTA cues are usually expressed within single sentences rather than spread across an entire turn. This motivates our later choice to *detect CTAs at the sentence level and then aggregate the results back to the turn level*.

We therefore use a binary distinction for reliability and interpretability:
- **CTA**: at least one utterance encouraging the listener to perform a specific action (for example visiting a website, subscribing, donating, or following a link).  
- **Non-CTA**: no such directive.

Binary classification is appropriate because CTAs are relatively sparse and often ambiguous. Developing fine-grained categories, such as differentiating promotional CTAs from engagement or civic CTAs, would require extensive annotation guidelines and substantial training data. Given limited annotation capacity, fine-grained distinctions would be unreliable, whereas a binary label is robust and fully sufficient for downstream analyses.

CTA identification is challenging: some cases are ambiguous or context dependent, CTAs directed at co-hosts rather than listeners ideally should not be counted, and softly phrased CTAs can be difficult to detect. Whisper transcription errors may obscure key cues, and very long turns can embed CTAs among unrelated material. Despite these limitations, our pipeline provides a practical and interpretable approach to identifying directive behaviour in podcast speech.


In [5]:
episodes_df = pd.read_parquet("data/episodes.parquet")
turns_df = pd.read_parquet("data/turns.parquet")
episode_lookup = {ep.mp3_url: ep for ep in episodes}
episodes = (episodes_df['mp3_url'].map(episode_lookup)).to_list()

#### Detecting CTA Language

There is no established off-the-shelf method for detecting CTA language in conversational podcast transcripts. [Recent work](https://doi.org/10.48550/arXiv.2409.02690) has explored prompting large language models (LLMs) to classify persuasive or CTA-like statements directly, showing that LLMs can detect subtle directive language. In practice, CTA cues almost always appear within individual sentences, not across entire turns, and some turns in podcast transcripts are extremely long. This means that running an LLM on full turns may detection quality and while often being infeasible due to context window limits. For this reason we will detect CTAs on sentence-level and then aggregate to the turn level. However, running an LLM on every sentence or turn in a large corpus is infeasible due to computational cost and time restrictions. 

To balance semantic quality and scalability, we adopt a hybrid, two-stage pipeline:

1. **Rule-based sentence-level flagging** to identify candidate CTA sentences using cheap lexical and structural cues.  
2. **LLM-based sentence-level classification** applied only to flagged sentences, followed by aggregation back to the turn level.

This design lets us use the LLM where it is most effective while avoiding the prohibitive cost of classifying all sentences.

#### Rule-Based CTA Flagging

In the first stage, we use a rule-based approach grounded in linguistic cues, lexical patterns, and URL-like expressions to flag sentences that might contain CTAs. Since no standard CTA lexicon exists for conversational speech, we developed domain-specific lexicons iteratively by:

- surveying common CTA verbs and phrases used in podcast intros, outros, and sponsor segments  
- reviewing marketing-oriented CTA verb lists (for example subscribe, follow, join, sign up, visit)  
- inspecting a sample of turns containing URLs or sponsor mentions  
- including multi-word expressions such as *"sign up"* and *"use code"* that frequently appear in spoken promotions  
- adding Whisper-style URL distortions such as *"dot com"* and *"slash slash"* to capture transcription artifacts

We then apply NLTK to split turns into sentences and use simple heuristics (CTA lexicon matches, URL-like patterns, and imperative-style structures) to flag sentences that are likely to contain CTA language. This stage is intentionally broad and inclusive: its primary goal is to capture as many potential CTAs as possible, even at the cost of false positives, so that the LLM in the next stage can focus on a much smaller subset of sentences.

The output of this step is a set of sentences marked as *candidate CTA sentences*, along with their turn indices, so that we can later aggregate back to the turn level.

In [10]:
import re
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
from nltk import word_tokenize, pos_tag
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm
tqdm.pandas()

In [7]:
CTA_VERBS = {
    "subscribe", "follow", "share", "like", "rate", "review",
    "visit", "check", "sign", "donate", "support", "join",
    "buy", "use", "download", "click", "call", "email",
    "register", "tell", "spread"
}

CTA_MULTIWORD = {
    "sign up", "log in", "check out"
}

CTA_PHRASES = {
    "use code", "promo code", "discount code", "free trial",
    "link in", "show notes","dot com", "dot org", "visit us at"
}

URL_PATTERN = re.compile(
    r"https?://\S+|"
    r"www\.\S+|"
    r"\b\w+\.(com|org|net|io|co)\b|"
    r"http\s+colon|"
    r"slash\s+slash",
    re.IGNORECASE
)

SECOND_PERSON = {"you", "your", "yours"}

def has_second_person(tokens):
    return any(t in SECOND_PERSON for t in tokens)

def has_cta_lexicon(tokens):
    """Return True only if a CTA verb appears in a CTA-like syntactic position."""
    if any(t in CTA_VERBS for t in tokens):
        return True
    
    return False

def is_imperative(tokens, pos_tags):
    """Very simple imperative heuristic:
    Sentence starts with a base form verb (VB) or 'please' followed by VB.
    """
    if not tokens:
        return False
    
    if pos_tags[0][1] == "VB":
        return True
    
    if tokens[0] == "please" and len(pos_tags) > 1 and pos_tags[1][1] == "VB":
        return True

    return False

In [8]:
# def is_cta_sentence(sentence):
#     """Return True if sentence contains CTA-like content."""
#     if not isinstance(sentence, str) or not sentence.strip():
#         return False

#     text = sentence.lower()
    
#     if URL_PATTERN.search(sentence):
#         return True
#     for phrase in CTA_PHRASES:
#         if phrase in text:
#             return True
#     for phrase in CTA_MULTIWORD:
#         if phrase in text:
#             return True

#     tokens = word_tokenize(text)
#     if any(tok in CTA_VERBS for tok in tokens):
#         return True
#     if tokens:
#         if tokens[0] in CTA_VERBS:
#             return True
#         if len(tokens) > 1 and tokens[1] in CTA_VERBS:
#             return True

#     return False

def is_cta_sentence(sentence):
    """Return True if sentence contains CTA-like content."""
    if not isinstance(sentence, str) or not sentence.strip():
        return False

    text = sentence.lower()
    if URL_PATTERN.search(text):
        return True
    
    for phrase in CTA_PHRASES:
        if phrase in text:
            return True
        
    for phrase in CTA_MULTIWORD:
        if phrase in text:
            return True

    # Token + POS-level cues
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    
    if is_imperative(tokens, pos_tags) and has_cta_lexicon(tokens):
        return True

    if has_cta_lexicon(tokens) and has_second_person(tokens):
        # exclude questions directed at co-hosts
        if not text.strip().endswith("?"):
            return True

    return False



def count_sentences(turn_text):
    """Count number of sentences in the turn."""
    if not isinstance(turn_text, str) or not turn_text.strip():
        return 0
    text = turn_text.strip()
    if not any(p in text for p in ".?!"):
        return 1
    return len(sent_tokenize(text))


def get_flagged_sentences(turn_text):
    """Return a list of sentences flagged as potential CTAs."""
    if not isinstance(turn_text, str) or not turn_text.strip():
        return []

    text = turn_text.strip()

    # No punctuation → treat whole turn as one sentence
    if not any(p in text for p in ".?!"):
        return [text] if is_cta_sentence(text) else []

    sentences = sent_tokenize(text)
    return [s for s in sentences if is_cta_sentence(s)]


def classify_turn_cta(turn_text):
    """Return True if any sentence in the turn contains a potential CTA."""
    if not isinstance(turn_text, str) or not turn_text.strip():
        return False

    text = turn_text.strip()

    if not any(p in text for p in ".?!"):
        return is_cta_sentence(text)

    sentences = sent_tokenize(text)
    return any(is_cta_sentence(s) for s in sentences)

In [11]:
turns_df['flagged'] = turns_df['clean_text'].progress_apply(
    lambda x: classify_turn_cta(x) if isinstance(x, str) else False
)
turns_df['flagged_sentences'] = turns_df['clean_text'].progress_apply(
    lambda x: get_flagged_sentences(x) if isinstance(x, str) else []
)
turns_df['num_flagged'] = turns_df['flagged_sentences'].apply(len)
turns_df['num_sentences'] = turns_df['clean_text'].progress_apply(
    lambda x: count_sentences(x)
)
turns_df.head()

  0%|          | 0/199491 [00:00<?, ?it/s]

  0%|          | 0/199491 [00:00<?, ?it/s]

  0%|          | 0/199491 [00:00<?, ?it/s]

,episode_title,turn_index,speaker,start_time,end_time,duration,raw_text,clean_text,is_removed,num_words,mp3_url,flagged,flagged_sentences,num_flagged,num_sentences
0,Best of SingOut SpeakOut No.3,0,SPEAKER_00,0.00,60.00,60.00,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,124,https://www.buzzsprout.com/783020/4252475-best...,True,[My will to connect with you and share the bes...,1,8
1,It's All Gone,0,SPEAKER_00,0.00,78.16,78.16,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,172,https://www.buzzsprout.com/783020/4165286-it-s...,True,[My will to connect with you and share the bes...,2,9
2,It's All Gone,1,SPEAKER_02,78.16,115.31,37.15,Music] [Music] [Music] [Music] [Music] [,None,True,0,https://www.buzzsprout.com/783020/4165286-it-s...,False,[],0,0
3,It's All Gone,2,SPEAKER_01,115.31,360.16,244.85,Music] [Music] [Music] [Music] [Music] [Music]...,None,True,0,https://www.buzzsprout.com/783020/4165286-it-s...,False,[],0,0
4,Today Is Yesterday,0,SPEAKER_05,0.00,36.99,36.99,I'm Simon Shapiro and this is Sing Out Speak ...,I'm Simon Shapiro and this is Sing Out Speak O...,False,77,https://www.buzzsprout.com/783020/3983942-toda...,True,[My will to connect with you and share the bes...,1,3


In [12]:
print(f"Proportion of turns flagged as CTA: {turns_df['flagged'].mean():.1%}")

valid = turns_df[turns_df['num_sentences'] > 0]
avg_cta_sentence_prop = (valid['num_flagged'] / valid['num_sentences']).mean()
print(f"Average flagged sentence proportion per turn: {avg_cta_sentence_prop:.1%}")

Proportion of turns flagged as CTA: 9.4%
Average flagged sentence proportion per turn: 4.6%


In [15]:
turns_df.to_parquet("data/turns_cta.parquet", index=False)
turns_df = pd.read_parquet("data/turns_cta.parquet")

In [16]:
turns_df[turns_df["episode_title"]=="Today Is Yesterday"]["flagged_sentences"]

4     [My will to connect with you and share the bes...
5                                                    []
6                                                    []
7     [But I know it can be tough when you're stuck ...
8                                                    []
9                                                    []
10                                                   []
11                                                   []
12                                                   []
13                                                   []
14                                                   []
15    [If you like the song, it's available at all t...
16                                                   []
Name: flagged_sentences, dtype: object

#### LLM-Based CTA Classification

In the second stage, we refine the rule-based predictions using a causal LLM. Although fine-tuning a smaller supervised classifier such as BERT or RoBERTa would likely produce the best performance, we do not have annotated data or the resources for supervised training. We therefore rely on few-shot prompting to leverage the semantic understanding of a causal LLM.

We use a few-shot prompt with representative CTA and non-CTA examples tailored to the model's expected input style to ensure consistent outputs.

The LLM is applied only to sentences flagged by the rule-based detector, which keeps computational cost manageable while preserving high recall. Each flagged sentence receives a binary CTA label, and these sentence-level predictions are then aggregated to the turn level. A turn is considered **CTA** if at least one of its sentences is classified as containing a CTA.

In [18]:
turns_df['cta_sentences'].sum() * (19 / 24) / 60 / 60

17.327604166666667

_____